# 3. LangChain Streaming - Respuestas en Tiempo Real (Moderno)

## Objetivos de Aprendizaje
- Comprender qué es el streaming y cuándo usarlo
- Implementar streaming con LangChain usando el enfoque moderno
- Manejar chunks de datos en tiempo real
- Construir interfaces de usuario reactivas
- Integrar memoria con streaming para chatbots contextuales

## ¿Qué es el Streaming?

El streaming permite recibir la respuesta del modelo **token por token** conforme se genera, en lugar de esperar a que termine completamente. Esto mejora significativamente la experiencia de usuario en aplicaciones interactivas.

### Ventajas del Streaming:
- **Percepción de velocidad**: El usuario ve progreso inmediato
- **Mejor UX**: Interfaces más reactivas e interactivas  
- **Engagement**: Mantiene la atención del usuario
- **Debugging**: Permite ver el proceso de generación

### Casos de Uso Ideales:
- Chatbots y asistentes conversacionales
- Generación de contenido largo
- Aplicaciones web interactivas
- Demostraciones en vivo

## Configuración Inicial

Selecciona el proveedor mediante la variable de entorno `PROVIDER`:
- `openrouter` (por defecto, modelos gratuitos)
- `google` (Google AI Studio, todos gratuitos)

In [ ]:
!pip install langchain langchain-openai langchain-core python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
from getpass import getpass
import time

load_dotenv()

# Determinar proveedor
PROVIDER = os.environ.get("PROVIDER", "openrouter").lower()
if PROVIDER not in ["openrouter", "google"]:
    raise ValueError("PROVIDER debe ser 'openrouter' o 'google'")

print(f"🔧 Proveedor seleccionado: {PROVIDER.upper()}")

# Configurar credenciales según proveedor
if PROVIDER == "openrouter":
    if not os.environ.get("OPENROUTER_API_KEY"):
        clave = getpass("Ingresa tu clave de OpenRouter: ")
        os.environ["OPENROUTER_API_KEY"] = clave
    BASE_URL = "https://openrouter.ai/api/v1"
    MODELO_DEFECTO = "minimax/minimax-m3:free"  # gratuito y disponible
else:  # google
    if not os.environ.get("GEMINI_API_KEY"):
        clave = getpass("Ingresa tu clave de Google AI Studio: ")
        os.environ["GEMINI_API_KEY"] = clave
    BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
    MODELO_DEFECTO = "gemini-3.6-flash"  # modelo actualizado

print(f"✅ Clave configurada para {PROVIDER.upper()}")
print(f"Modelo por defecto: {MODELO_DEFECTO}")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
import warnings
warnings.filterwarnings('ignore')

def crear_llm(modelo=MODELO_DEFECTO, temperature=0.7, max_tokens=500):
    """Crea un LLM con la configuración del proveedor seleccionado."""
    if PROVIDER == "openrouter":
        api_key = os.environ.get("OPENROUTER_API_KEY")
    else:
        api_key = os.environ.get("GEMINI_API_KEY")

    if not api_key:
        raise ValueError(f"❌ Clave para {PROVIDER.upper()} no encontrada.")

    kwargs = {
        "base_url": BASE_URL,
        "api_key": api_key,
        "model": modelo,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if PROVIDER == "openrouter":
        kwargs["default_headers"] = {
            "HTTP-Referer": "http://localhost:8000",
            "X-Title": "LangChain Streaming",
        }
    return ChatOpenAI(**kwargs)

llm = crear_llm()

# Prueba de conexión
response = llm.invoke([HumanMessage(content="Hola, responde 'OK' si me escuchas.")])
print(f"✅ Conexión exitosa con {PROVIDER.upper()}")
print(f"Respuesta: {response.content}")

## Streaming Básico

El método `.stream()` devuelve un generador que produce chunks de texto conforme se generan.

In [ ]:
def streaming_basico():
    prompt = "Cuéntame una historia corta sobre un programador que descubre la magia en el código"

    print("=== STREAMING EN TIEMPO REAL ===")
    print("Generando respuesta...")
    print("-" * 50)

    try:
        # stream() devuelve un generador de chunks
        full_response = ""
        for chunk in llm.stream([HumanMessage(content=prompt)]):
            content = chunk.content
            print(content, end="", flush=True)
            full_response += content
            time.sleep(0.01)  # Pausa para efecto visual

        print("\n" + "-" * 50)
        print(f"✓ Streaming completado ({len(full_response)} caracteres)")

        # Mostrar metadatos (si están disponibles)
        if hasattr(chunk, 'response_metadata'):
            print(f"Modelo: {chunk.response_metadata.get('model', 'N/A')}")

    except Exception as e:
        print(f"✗ Error en streaming: {e}")

streaming_basico()

## Comparación: Streaming vs No-Streaming

Veamos la diferencia en experiencia de usuario entre ambos enfoques.

In [7]:
def comparar_streaming():
    # Modelo sin streaming (usamos el mismo, pero sin llamar a stream)
    prompt = "Escribe un párrafo sobre las ventajas de la programación en Python"

    print("=== COMPARACIÓN: STREAMING vs NO-STREAMING ===\n")

    # 1. Sin streaming
    print("1. SIN STREAMING:")
    print("-" * 20)
    print("Esperando respuesta completa...")

    start_time = time.time()
    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        end_time = time.time()

        print(f"\n[Respuesta recibida después de {end_time - start_time:.2f} segundos]")
        print(response.content)

    except Exception as e:
        print(f"Error: {e}")

    print("\n" + "="*60 + "\n")

    # 2. Con streaming
    print("2. CON STREAMING:")
    print("-" * 18)
    print("Respuesta en tiempo real:")

    start_time = time.time()
    try:
        for chunk in llm.stream([HumanMessage(content=prompt)]):
            print(chunk.content, end="", flush=True)
            time.sleep(0.03)  # Efecto visual

        end_time = time.time()
        print(f"\n\n[Streaming completado en {end_time - start_time:.2f} segundos]")

    except Exception as e:
        print(f"Error: {e}")

    print("\n" + "="*60)
    print("OBSERVACIONES:")
    print("- Sin streaming: El usuario espera sin feedback")
    print("- Con streaming: El usuario ve progreso inmediato")
    print("- La percepción de velocidad es mucho mejor con streaming")
    print("- Ideal para respuestas largas (>100 palabras)")

comparar_streaming()

=== COMPARACIÓN: STREAMING vs NO-STREAMING ===

1. SIN STREAMING:
--------------------
Esperando respuesta completa...

[Respuesta recibida después de 6.37 segundos]
## Ventajas de la programación en Python

Python se ha consolidado como uno de los lenguajes de programación más populares del mundo gracias a sus numerosas ventajas, entre las que destaca su **sintaxis clara y legible**, que se asemeja al lenguaje natural y permite a los desarrolladores escribir código de forma intuitiva y concisa, reduciendo el tiempo de aprendizaje especialmente para principiantes. Además, cuenta con una enorme biblioteca estándar y un ecosistema muy amplio de paquetes y frameworks especializados en campos como el desarrollo web (Django, Flask), la ciencia de datos (Pandas, NumPy), el aprendizaje automático (TensorFlow, PyTorch) o la automatización, lo que lo convierte en una herramienta **versátil y multiplataforma** capaz de adaptarse a prácticamente cualquier proyecto. Su naturaleza **interpretada y 

## Chatbot Interactivo con Streaming y Memoria

Este chatbot mantiene contexto gracias a `RunnableWithMessageHistory`. Además, muestra las respuestas en streaming.

In [9]:
from langchain_core.runnables import RunnablePassthrough

# Configurar memoria
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Plantilla de prompt con historial
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil y amigable especializado en tecnología. Respondes de manera clara y concisa, y siempre intentas ser educativo."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# Cadena: prompt → llm → parser
chain = prompt_template | llm | StrOutputParser()

# Envolver con memoria
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

def chatbot_streaming():
    print("=== CHATBOT CON STREAMING Y MEMORIA ===")
    print("Escribe 'salir' para terminar la conversación\n")

    session_id = "chatbot_streaming"

    while True:
        user_input = input("\n🧑 Tú: ")

        if user_input.lower() in ['salir', 'exit', 'quit']:
            print("\n👋 ¡Hasta luego!")
            break

        if not user_input.strip():
            continue

        print("\n🤖 Asistente: ", end="", flush=True)

        try:
            full_response = ""
            # Usamos stream directamente sobre la cadena con memoria
            for chunk in chain_with_memory.stream(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            ):
                print(chunk, end="", flush=True)
                full_response += chunk
                time.sleep(0.02)
            print()  # Nueva línea al final

        except KeyboardInterrupt:
            print("\n\n⏸️ Interrumpido por el usuario")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")

print("💡 Para ejecutar el chatbot, descomenta la siguiente línea:")
chatbot_streaming()

💡 Para ejecutar el chatbot, descomenta la siguiente línea:
=== CHATBOT CON STREAMING Y MEMORIA ===
Escribe 'salir' para terminar la conversación


🧑 Tú: Hola

🤖 Asistente: ¡Hola! 👋 ¿En qué puedo ayudarte hoy?

🧑 Tú: quiero saber cunado es el cambio de hora en chile

🤖 Asistente: El **cambio de hora en Chile** está regulado por el Ministerio de Energía y normalmente sigue este calendario:

## 🇧🇷 ¡Ups, disculpá! Me equivoqué de país

El cambio de hora es en **Chile**, no en Brasil. Acá te explico:

## Cambio de hora en Chile 🇨🇱

En Chile se aplican **dos cambios de hora al año**:

### 1. Cambio de horario de verano → invierno
- **Cuándo:** Primer sábado de abril
- **Qué pasa:** A las 00:00 los relojes se atrasan **1 hora**
- **Ejemplo 2025:** Sábado 5 de abril

### 2. Cambio de horario de invierno → verano
- **Cuándo:** Primer sábado de septiembre
- **Qué pasa:** A las 00:00 los relojes se adelantan **1 hora**
- **Ejemplo 2025:** Sábado 6 de septiembre

## Zonas horarias de Chile
- **Chi

## Streaming Avanzado con Análisis de Chunks

Podemos procesar cada chunk individualmente para crear experiencias más sofisticadas (contar palabras, medir progreso, etc.).

In [10]:
def streaming_avanzado():
    prompt = "Explica qué es la inteligencia artificial y cómo funciona el machine learning"

    print("=== STREAMING AVANZADO CON ANÁLISIS ===")
    print("Analizando chunks conforme llegan...\n")

    chunk_count = 0
    total_content = ""
    words_processed = 0

    try:
        for chunk in llm.stream([HumanMessage(content=prompt)]):
            chunk_count += 1
            content = chunk.content
            total_content += content

            if content.strip():
                words_in_chunk = len(content.split())
                words_processed += words_in_chunk

            # Mostrar progreso cada 10 chunks
            if chunk_count % 10 == 0:
                print(f"\n[Progreso: {chunk_count} chunks, ~{words_processed} palabras]\n")

            print(content, end="", flush=True)
            time.sleep(0.02)

        # Estadísticas finales
        print(f"\n\n=== ESTADÍSTICAS FINALES ===")
        print(f"Total de chunks: {chunk_count}")
        print(f"Palabras aproximadas: {words_processed}")
        print(f"Caracteres totales: {len(total_content)}")
        if chunk_count > 0:
            print(f"Promedio chars/chunk: {len(total_content)/chunk_count:.1f}")

    except Exception as e:
        print(f"\n✗ Error: {e}")

streaming_avanzado()

=== STREAMING AVANZADO CON ANÁLISIS ===
Analizando chunks conforme llegan...

# Inteligencia Artificial y Machine Learning

## ¿Qué es la Inteligencia Artificial?

La **Inteligencia Artificial (IA)**
[Progreso: 10 chunks, ~22 palabras]

 es una rama de la informática que busca crear sistemas capaces de realizar tareas que, cuando las hacen humanos
[Progreso: 20 chunks, ~41 palabras]

, requieren inteligencia. Estas tareas incluyen:

- Razonamiento lógico
- Aprendizaje
- Per
[Progreso: 30 chunks, ~57 palabras]

cepción (visión, voz)
- Comprensión del lenguaje
- Toma de decisiones

---

## ¿Qué es el
[Progreso: 40 chunks, ~78 palabras]

 Machine Learning?

El **Machine Learning (ML)** o *aprendizaje automático* es una subdisciplina de la IA que permite a las máquinas
[Progreso: 50 chunks, ~101 palabras]

 **aprender a partir de datos** sin ser programadas con reglas explícitas.
[Progreso: 60 chunks, ~116 palabras]

 En lugar de escribir instrucciones paso a paso, se les dan ejemplos para

## Consideraciones Técnicas del Streaming

### Cuándo Usar Streaming:
✅ **SÍ usar streaming:**
- Respuestas largas (>100 tokens)
- Aplicaciones interactivas
- Chatbots y asistentes
- Demostraciones en vivo
- Cuando la UX es prioritaria

❌ **NO usar streaming:**
- Respuestas muy cortas
- Procesamiento batch
- APIs de backend sin interfaz
- Cuando necesitas la respuesta completa antes de procesar

### Mejores Prácticas:
1. **Manejo de errores**: Siempre incluye try/catch específico para streaming.
2. **Indicadores visuales**: Muestra progreso al usuario (spinners, barras).
3. **Cancelación**: Permite al usuario interrumpir el streaming.
4. **Buffer management**: Para interfaces web, considera buffering.
5. **Memoria**: Combina streaming con memoria para experiencias contextuales.

## Ejercicios Prácticos

### Ejercicio 1: Indicador de Progreso
Modifica el streaming avanzado para mostrar un porcentaje de completitud estimado.

### Ejercicio 2: Streaming con Filtros
Implementa streaming que resalte palabras clave (ej: "IA", "machine learning") en mayúsculas.

### Ejercicio 3: Chatbot Mejorado
Extiende el chatbot con:
- Comandos especiales (/help, /clear, /history)
- Diferentes personalidades (cambiando el system prompt)
- Persistencia de historial entre sesiones

## Conceptos Clave Aprendidos

1. **Streaming** mejora la percepción de velocidad y la UX.
2. **Chunks** se procesan individualmente en tiempo real.
3. **Memoria + Streaming** permite chatbots contextuales y reactivos.
4. **Implementación** requiere manejo cuidadoso de generadores y errores.
5. **Casos de uso** específicos donde streaming aporta valor.

## Próximos Pasos

En el siguiente notebook exploraremos la **memoria en LangChain** en profundidad, incluyendo diferentes tipos de memoria y su integración con cadenas y agentes.